# Variational Quantum Classifier for Binary Classification

**Authors:** Koło Naukowe Axion  
**Dataset:** Banknote Authentication (UCI ML Repository)  
**Framework:** Qiskit Machine Learning + PyTorch

## Abstract

This notebook demonstrates a hybrid Variational Quantum Circuit (VQC) for binary classification using the banknote authentication dataset. The model leverages:

- **Feature encoding**: Angle encoding via RY rotations
- **Parametrized quantum circuit (VQC)**: Hardware-efficient ansatz with ring topology entanglement
- **Measurement**: Expectation value of the Pauli-Z observable on the first qubit
- **Classical optimizer**: Adam optimizer via PyTorch's automatic differentiation

The hybrid architecture uses Qiskit's `EstimatorQNN` bridged to PyTorch via `TorchConnector`, enabling seamless gradient-based training on a simulated quantum computer.

## 1. Environment Setup

This section handles dependency installation and imports. For reproducibility, all package versions should be pinned in a production environment.

### 1.1 Package Installation (Optional)

Set `INSTALL_DEPS = True` if running in a fresh environment. For production use, pin specific versions.

In [1]:
# Optional: Install dependencies if not already present
INSTALL_DEPS = False

if INSTALL_DEPS:
    import sys
    import subprocess
    
    packages = [
        'numpy',
        'scikit-learn',
        'ucimlrepo',
        'qiskit',
        'qiskit-machine-learning',
        'torch',
        'matplotlib'
    ]
    
    for pkg in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

### 1.2 Imports

In [2]:
# Standard library
import os
import random
import copy 

# Third-party: Scientific computing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Third-party: Machine learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score
from ucimlrepo import fetch_ucirepo

# Third-party: Quantum computing
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms.utils import algorithm_globals
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN


## 2. Reproducibility and Random Seed Control

In [3]:
def set_random_seed(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across numpy, PyTorch, and Python's random module.
    
    Parameters
    ----------
    seed : int
        Random seed value (default: 42)
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False    
    algorithm_globals.random_seed = seed

# Set global seed
RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)

## 3. Quantum Circuit Architecture

The quantum model consists of two components:
1. **Feature map** (angle encoding): Encodes classical data into quantum states
2. **Ansatz** (trainable circuit): Parametrized quantum circuit with learnable weights

### 3.1 Parametrized Ansatz

Ansatz with alternating rotation and entanglement layers. The circuit uses a ring topology for entanglement, alternating between forward (CRX) and reverse (CRY) connections.

In [4]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    """Weights when only the last macro-layer uses the q0-incident reverse trim."""
    m = depth // 2
    if m == 0:
        return 0
    full = 4 * n_qubits
    last = 3 * n_qubits + 2
    return (m - 1) * full + last


def odra_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    # Full reverse (4n) on all macro-layers except the last; there trim reverse to q0 edges only (3n+2).
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    p = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[p + i], i)
        p += n_qubits

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.rz(theta[p + i], target)
            qc.cz(control, target)
        p += n_qubits

        for i in range(n_qubits):
            qc.rx(theta[p + i], i)
        p += n_qubits

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + k], target)
                qc.cz(control, target)
            p += 2
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

    assert p == len(theta)
    return qc

### 3.2 Hybrid Variational Quantum Circuit

The `HybridModel` class implements a VQC by integrating the quantum circuit with PyTorch's autograd system via Qiskit's `TorchConnector`. This enables gradient-based optimization of quantum parameters using classical optimizers.

In [5]:
class HybridModel(nn.Module):
    """
    Hybrid Variational Quantum Circuit (VQC) for binary classification.
    
    The model combines:
    1. Angle encoding feature map (classical data → quantum state)
    2. Parametrized ansatz (trainable quantum circuit)
    3. Observable measurement (quantum state → classical expectation value)
    4. PyTorch integration via TorchConnector (enables backpropagation)
    
    Parameters
    ----------
    ansatz_circuit : QuantumCircuit
        Parametrized quantum circuit with trainable weights
    num_qubits : int
        Number of qubits (must match feature dimension)
    
    Attributes
    ----------
    qnn : EstimatorQNN
        Qiskit's EstimatorQNN that computes expectation values
    quantum_layer : TorchConnector
        PyTorch-compatible wrapper enabling gradient computation
    
    Notes
    -----
    - **Feature encoding**: RY(x_i) on qubit i encodes feature x_i
    - **Observable**: Pauli-Z on qubit 0, measuring spin in computational basis
    - **Output range**: [-1, +1] (expectation value of Z operator)
    - **Gradient method**: Parameter shift rule for quantum gradients
    - **Simulator**: StatevectorEstimator (change for real quantum hardware)
    """
    
    def __init__(self, ansatz_circuit, num_qubits):
        super().__init__()
        
        # Create angle encoding feature map
        self.feature_map = self._create_angle_encoding(num_qubits)
        
        # Compose full quantum circuit: feature_map → ansatz
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)
        
        # Separate input parameters (from feature map) and weight parameters (from ansatz)
        # This distinction is crucial for EstimatorQNN to correctly handle data vs. trainable weights
        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)
        
        # Define observable: measure Z on qubit 0 (identity on other qubits)
        # Pauli string ordering: rightmost character = qubit 0
        # Example for 5 qubits: "IIIIZ" measures Z on q0, I on q1-q4
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
        
        # Initialize statevector simulator for noiseless quantum simulation
        # NOTE: Replace with Sampler or real backend for quantum hardware deployment
        estimator = StatevectorEstimator()
        

        gradient = ParamShiftEstimatorGradient(estimator)
        # Create variational quantum circuit using EstimatorQNN
        # EstimatorQNN computes <ψ|O|ψ> where |ψ> = ansatz(weights)|feature_map(x)>
        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient
        )
        
        # Wrap the VQC as a PyTorch module
        # TorchConnector bridges Qiskit and PyTorch autograd systems,
        # allowing standard PyTorch optimizers (SGD, Adam, etc.) to train quantum parameters
        self.quantum_layer = TorchConnector(self.qnn)
    
    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        """
        Create angle encoding feature map: |0⟩ → RY(x₀) ⊗ RY(x₁) ⊗ ... ⊗ RY(xₙ) |0⟩
        
        Each classical feature x_i ∈ [0, π] is encoded as a rotation angle on qubit i.
        This maps the feature vector to the amplitude of the quantum state.
        
        Parameters
        ----------
        num_qubits : int
            Number of qubits (and features)
        
        Returns
        -------
        QuantumCircuit
            Feature map circuit with n_qubits input parameters
        """
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector('x', num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the hybrid variational quantum circuit.
        
        Parameters
        ----------
        x : torch.Tensor
            Input features, shape (batch_size, num_qubits)
        
        Returns
        -------
        torch.Tensor
            Expectation values, shape (batch_size, 1), range [-1, +1]
        """
        return self.quantum_layer(x)

## 4. Training Configuration and Data Loading

In [6]:
# Hyperparameters
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.01
NUM_QUBITS = 5
ANSATZ_DEPTH = 4
K_FOLDS = 5

## 5. Model Training

We train the hybrid VQC using Mean Squared Error (MSE) loss and the Adam optimizer. The training loop evaluates both training and test performance at each epoch.

In [7]:
# Define relative paths based on the notebook's location
DATA_DIR = "../../../../Data"
WEIGHTS_DIR = f"../../../Weights/depth {ANSATZ_DEPTH}/Odra"

# Validate if the data directory exists
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Data directory not found at: {DATA_DIR}. Check relative path.")

# Create the base weights directory if it doesn't exist
os.makedirs(WEIGHTS_DIR, exist_ok=True)

all_y_true = []
all_y_pred = []

fold_final_accuracies = []
fold_final_f1s = []

for fold in range(1, K_FOLDS + 1):
    print(f"\n--- FOLD {fold}/{K_FOLDS} ---")

    # Ensure deterministic behavior and independent initialization for this specific fold
    set_random_seed(RANDOM_SEED)

    # 1. Load data from the Data directory
    fold_data_dir = f"{DATA_DIR}/fold_{fold}"
    train_csv = f"{fold_data_dir}/train_data.csv"
    test_csv = f"{fold_data_dir}/test_data.csv"
    
    try:
        train_df = pd.read_csv(train_csv)
        test_df = pd.read_csv(test_csv)
    except FileNotFoundError:
        print(f"Error: Could not find data files at {fold_data_dir}")
        break

    X_train_scaled = train_df.drop('target', axis=1).values
    y_train = train_df['target'].values
    X_test_scaled = test_df.drop('target', axis=1).values
    y_test = test_df['target'].values

    # Convert to PyTorch tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

    # Create an explicit generator for the DataLoader to guarantee deterministic shuffling
    g = torch.Generator()
    g.manual_seed(RANDOM_SEED + fold)

    # Create DataLoader for batched training
    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=BATCH_SIZE, 
        shuffle=True,
        generator=g
    )

    # Ensure the directory for storing weights exists inside the Weights structure
    model_fold_dir = f"{WEIGHTS_DIR}/fold_{fold}"
    os.makedirs(model_fold_dir, exist_ok=True)

    # Initialize a new ansatz and hybrid model for the current fold
    current_ansatz = odra_ansatz(NUM_QUBITS, ANSATZ_DEPTH)
    model = HybridModel(current_ansatz, NUM_QUBITS)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_function = torch.nn.MSELoss()

    epoch_20_weights = None

    # Training loop (Clean: No evaluation on the test set during training)
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()           
            output = model(X_batch)         
            loss = loss_function(output, y_batch) 
            loss.backward()                 
            optimizer.step()                
            epoch_loss += loss.item()
            
        # Capture the model weights exactly after the 20th epoch (index 19)
        if epoch == 19:
            epoch_20_weights = copy.deepcopy(model.state_dict())

    # Calculate final metrics using the model from the last epoch (Epoch 30)
    # This is the ONLY time the model sees the test set, preventing data leakage
    model.eval()
    with torch.no_grad():
        preds = (torch.round(model(X_test_tensor), decimals=5) > 0).float() * 2 - 1
        y_true_np = y_test_tensor.numpy().flatten()
        preds_np = preds.numpy().flatten()
        
        all_y_true.extend(y_true_np)
        all_y_pred.extend(preds_np)
        
        final_acc = accuracy_score(y_true_np, preds_np)
        final_f1 = f1_score(y_true_np, preds_np, pos_label=1)

    fold_final_accuracies.append(final_acc)
    fold_final_f1s.append(final_f1)

    # Define paths for saving weights from Epoch 20 and Epoch 30
    path_epoch_20 = f"{model_fold_dir}/Odra_fold_{fold}_depth_{ANSATZ_DEPTH}_epoch_20_weights.pth"
    path_epoch_30 = f"{model_fold_dir}/Odra_fold_{fold}_depth_{ANSATZ_DEPTH}_epoch_30_weights.pth"
    
    # Save both checkpoints to the designated Weights directory
    if epoch_20_weights is not None:
        torch.save(epoch_20_weights, path_epoch_20)
    torch.save(model.state_dict(), path_epoch_30)
    
    print(f"Results fold {fold} (Epoch 30):")
    print(f"  Accuracy: {final_acc:.4f} | F1: {final_f1:.4f}")
    print(f"  -> Saved Epoch 20 weights to: {path_epoch_20}")
    print(f"  -> Saved Epoch 30 weights to: {path_epoch_30}")


--- FOLD 1/5 ---
Results fold 1 (Epoch 30):
  Accuracy: 0.8364 | F1: 0.7761
  -> Saved Epoch 20 weights to: ../../../Weights/depth 4/Odra/fold_1/Odra_fold_1_depth_4_epoch_20_weights.pth
  -> Saved Epoch 30 weights to: ../../../Weights/depth 4/Odra/fold_1/Odra_fold_1_depth_4_epoch_30_weights.pth

--- FOLD 2/5 ---
Results fold 2 (Epoch 30):
  Accuracy: 0.8945 | F1: 0.8722
  -> Saved Epoch 20 weights to: ../../../Weights/depth 4/Odra/fold_2/Odra_fold_2_depth_4_epoch_20_weights.pth
  -> Saved Epoch 30 weights to: ../../../Weights/depth 4/Odra/fold_2/Odra_fold_2_depth_4_epoch_30_weights.pth

--- FOLD 3/5 ---
Results fold 3 (Epoch 30):
  Accuracy: 0.9051 | F1: 0.8943
  -> Saved Epoch 20 weights to: ../../../Weights/depth 4/Odra/fold_3/Odra_fold_3_depth_4_epoch_20_weights.pth
  -> Saved Epoch 30 weights to: ../../../Weights/depth 4/Odra/fold_3/Odra_fold_3_depth_4_epoch_30_weights.pth

--- FOLD 4/5 ---
Results fold 4 (Epoch 30):
  Accuracy: 0.8759 | F1: 0.8522
  -> Saved Epoch 20 weights to: 